# 03 — Link Analysis in Spark: PageRank + HITS
**Exam mapping (P3):** 3a PageRank top-25 [10], 3b HITS hubbiness+authority top-25 [10], 3c three advice-to-CEO backed by results [5].

**Key constraint:** built-in Spark only ⇒ **no GraphFrames**. You implement the iteration yourself with DataFrame joins (PageRank) and RDD message-passing (HITS). Both are just repeated "spread score along edges → sum at destination → normalize."

Workflow每 subproblem: load → filter to links → inspect → iterate → top-25 → interpret.

In [ ]:
# ---- Spark session (exam-safe: local master, built-in libs only) ----
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (SparkSession.builder
         .appName("sds-link-analysis")
         .master("local[*]")
         .config("spark.sql.shuffle.partitions", "16")
         .config("spark.ui.enabled", "false")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

In [ ]:
# ---- load the clickstream, keep only links (the exam says "consider only links") ----
DATA_PATH = "data/pageviews_practice.csv"     # <-- exam: "pageviews.csv"

raw = (spark.read.csv(DATA_PATH, sep="\t", header=False, inferSchema=True)
       .toDF("src", "dst", "type", "count"))

# ALWAYS inspect first: schema + type distribution (catches the messy-TSV trap)
raw.groupBy("type").count().orderBy(F.desc("count")).show()

links = (raw.filter(F.col("type") == "link")
            .select("src", "dst").dropDuplicates().cache())
n_edges = links.count()
nodes = (links.select(F.col("src").alias("node"))
         .union(links.select(F.col("dst").alias("node"))).distinct().cache())
N = nodes.count()
print(f"link edges = {n_edges:,} | nodes = {N:,}")

## 3a — PageRank via power iteration (DataFrame)
**Update:** `PR(v) = (1−d)/N + d · Σ_{u→v} PR(u)/outdeg(u)`, damping **d=0.85**, ~10–20 iterations.
**Two things graders check:** teleport term present (handles dead-ends/spider-traps), and **dangling mass** (nodes with no out-links) redistributed — otherwise rank leaks and ranks don't sum to 1. The cell below folds dangling mass into the teleport.

In [ ]:
d = 0.85
NUM_ITERS = 15

outdeg = links.groupBy("src").agg(F.count("*").alias("outdeg")).cache()
ranks = nodes.withColumn("rank", F.lit(1.0 / N))          # uniform init

for it in range(NUM_ITERS):
    # contribution each source sends along each out-edge
    contribs = (links.join(ranks, links.src == ranks.node)
                     .join(outdeg, "src")
                     .select(F.col("dst").alias("node"),
                             (F.col("rank") / F.col("outdeg")).alias("c")))
    incoming = contribs.groupBy("node").agg(F.sum("c").alias("inc"))

    # dangling mass = total rank on nodes with no out-links (leaks without this)
    dangling = (ranks.join(outdeg, ranks.node == outdeg.src, "left_anti")
                     .agg(F.sum("rank")).first()[0]) or 0.0

    ranks = (nodes.join(incoming, "node", "left").fillna(0.0, subset=["inc"])
                  .withColumn("rank",
                     (1 - d) / N + d * (F.col("inc") + dangling / N))
                  .select("node", "rank"))
    if it % 5 == 4:
        s = ranks.agg(F.sum("rank")).first()[0]
        print(f"iter {it+1:2d}: rank sum = {s:.6f} (should stay ~1.0)")

top25_pr = ranks.orderBy(F.desc("rank"), F.asc("node")).limit(25)
print("\nTop-25 by PageRank:")
top25_pr.show(25, truncate=False)

### What to report (3a)
- Damping d=0.85, N nodes, K iterations; rank sum stays ≈1 (proves dangling mass handled).
- The top nodes are the network's authorities-by-random-surfer — quote the top 3–5 and note they are broad hub topics, consistent with a heavy-tailed in-degree.
- Interpretation ties to 3c: these are the pages a random reader lands on most.

## 3b — HITS (hubs & authority) via RDD message-passing
**Updates:** `auth(v) = Σ_{u→v} hub(u)`, then `hub(u) = Σ_{u→v} auth(v)`, **L2-normalize each vector every iteration**. No teleport needed. ~5–10 iterations. HITS is the eigenvectors of AᵀA (authority) and AAᵀ (hub).

RDDs are the clean way here: `edges` as `(src,dst)`, join scores, `reduceByKey` to sum.

**Scalability lesson (this bit the real exam):** iterating RDD joins without materializing grows the lineage DAG unboundedly — every `.sum()`/`.take()` recomputes from scratch and it hangs. Fix: **persist + checkpoint + count each new score vector every iteration** to cut lineage. This is the SDS analogue of "don't leak dangling mass" — a correctness-of-scale detail graders reward.

In [ ]:
from math import sqrt
from pyspark import StorageLevel

spark.sparkContext.setCheckpointDir("/tmp/hits_ckpt")   # required for .checkpoint()

edges     = links.rdd.map(lambda r: (r["src"], r["dst"])).persist(StorageLevel.MEMORY_AND_DISK)
edges_rev = edges.map(lambda e: (e[1], e[0])).persist(StorageLevel.MEMORY_AND_DISK)
node_rdd  = edges.flatMap(lambda e: [e[0], e[1]]).distinct().persist(StorageLevel.MEMORY_AND_DISK)
edges.count(); edges_rev.count(); node_rdd.count()      # materialize graph structure once

auth = node_rdd.map(lambda n: (n, 1.0))
hub  = node_rdd.map(lambda n: (n, 1.0))

def l2norm(rdd):
    s = sqrt(rdd.map(lambda kv: kv[1] ** 2).sum()) or 1.0
    return rdd.mapValues(lambda v: v / s)

NUM_ITERS_HITS = 5     # top of the ranking stabilizes quickly
for it in range(NUM_ITERS_HITS):
    # authority(v) = Σ hub(u) over edges u->v   :  join hub on src, emit (dst, hub_src), sum
    auth = l2norm(edges.join(hub).map(lambda kv: (kv[1][0], kv[1][1]))
                       .reduceByKey(lambda a, b: a + b)).persist(StorageLevel.MEMORY_AND_DISK)
    auth.checkpoint(); auth.count()                     # cut lineage + materialize
    # hub(u) = Σ authority(v) over edges u->v    :  join auth on dst, emit (src, auth_dst), sum
    hub = l2norm(edges_rev.join(auth).map(lambda kv: (kv[1][0], kv[1][1]))
                          .reduceByKey(lambda a, b: a + b)).persist(StorageLevel.MEMORY_AND_DISK)
    hub.checkpoint(); hub.count()

hits = hub.join(auth)  # node -> (hub, auth)
top_hubs = hits.takeOrdered(25, key=lambda x: (-x[1][0], x[0]))
top_auth = hits.takeOrdered(25, key=lambda x: (-x[1][1], x[0]))

print("Top-25 by HUBBINESS:")
for node,(h,a) in top_hubs[:25]:
    print(f"  {node:22s} hub={h:.5f} auth={a:.5f}")
print("\nTop-25 by AUTHORITY:")
for node,(h,a) in top_auth[:25]:
    print(f"  {node:22s} hub={h:.5f} auth={a:.5f}")

### What to report (3b)
- Authority = pages many good hubs point to (broad destinations); hubbiness = pages that point to many good authorities (navigational sources). L2-normalized, K iterations, no teleport.
- Note pages high on **both** (entry points that are also referenced). Contrast the two top-25 lists — they should differ, and that difference is the point.

## 3c — Advice to the Wikimedia CEO (each backed by a result)
Formula: **claim → cite the specific 3a/3b result → action.** Three that always work on this data:
1. **Protect the top-PageRank / top-authority pages** (e.g. United_States, India, World_War_II): they are core navigation reference points, so vandalism there affects the most readers → prioritize editorial review + vandalism detection on them.
2. **Improve navigation on top-hub pages** (which differ from top-authority): they are gateways users pass through → keep their outgoing link sections well-organized and current.
3. **Cross-link the dominant topic cluster** (geopolitics/history dominate both rankings) → strengthen "see also"/templates among these major pages to deepen exploration.
Back each with the actual names/scores your cells printed — do not write generic advice.

In [ ]:
edges.unpersist(); edges_rev.unpersist(); links.unpersist(); nodes.unpersist()
spark.stop()
print("done")

## Link-analysis gotchas (exam bait)
- PageRank **needs teleport** (d=0.85) or dead-ends/spider-traps break it; **dangling mass** must be redistributed or ranks leak (sum < 1).
- HITS needs **no** teleport; **L2-normalize every iteration** or scores blow up; hub & authority are distinct — report both.
- "Built-in Spark only" ⇒ no GraphFrames: implement the iteration by hand (shown above).
- Quote the numbers your code printed — write-up figures must match executed output.